In [10]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# =====================================================================
# 1. DATA LOADING
# =====================================================================
print("--- Step 1: Loading Dataset ---")
# Load your single CSV file
df = pd.read_csv("adult.csv")
print(f"Original Dataset Shape: {df.shape}\n")


# =====================================================================
# 2. DATA CLEANING (Handling Hidden Missing Values)
# =====================================================================
print("--- Step 2: Cleaning the Data ---")

# Step A: Strip whitespaces and standardize hidden '?' missing values to true NaNs
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
df = df.replace("?", np.nan)

print("Actual missing values detected per column:")
missing_summary = df.isnull().sum()
print(missing_summary[missing_summary > 0])
print()

# Step B: Impute Missing Categorical Features using the mode (most frequent)
for col in ["workclass", "occupation", "native-country"]:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])


# =====================================================================
# 3. DATA TRANSFORMATION
# =====================================================================
print("--- Step 3: Transforming Data ---")

# A. Binary Feature Mapping
# Ensure the income target is uniformly mapped to 0 and 1
df["income"] = df["income"].astype(str).str.replace(".", "", regex=False)
df["income"] = df["income"].map({"<=50K": 0, ">50K": 1})
df["gender"] = df["gender"].map({"Male": 0, "Female": 1})

# B. Feature Engineering
# Create a net capital flow variable
df["capital_net"] = df["capital-gain"] - df["capital-loss"]

# C. Log Transformation (Fixes skewness on net financial gain/loss)
df["capital_net_log"] = np.sign(df["capital_net"]) * np.log1p(
    df["capital_net"].abs()
)

# D. One-Hot Encoding for categorical features
categorical_cols = ["workclass", "marital-status", "occupation", "relationship", "race"]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

# E. Feature Scaling
scaler = StandardScaler()
scale_targets = ["age", "hours-per-week", "educational-num"]
df[[f"{c}_scaled" for c in scale_targets]] = scaler.fit_transform(
    df[scale_targets]
)

print("Data transformation complete.\n")


# =====================================================================
# 4. DATA REDUCTION
# =====================================================================
print("--- Step 4: Reducing Data ---")

# Strategy A: Feature Selection (Drop uninformative, noisy, or raw duplicated columns)
columns_to_drop = [
    "fnlwgt",
    "education",
    "native-country",
    "capital-gain",
    "capital-loss",
    "capital_net",
    "age",
    "hours-per-week",
    "educational-num",
]
reduced_df = df.drop(columns=columns_to_drop)
print(f"Shape after manual feature selection: {reduced_df.shape}")

# Strategy B: Dimensionality Reduction via PCA
scaled_cols = [c for c in reduced_df.columns if "_scaled" in c]
pca = PCA(n_components=2)
pca_features = pca.fit_transform(reduced_df[scaled_cols])

# Append PCA features and drop raw continuous variables
reduced_df["Demographic_PCA_1"] = pca_features[:, 0]
reduced_df["Demographic_PCA_2"] = pca_features[:, 1]
reduced_df = reduced_df.drop(columns=scaled_cols)

print(f"Final reduced dataframe shape: {reduced_df.shape}\n")


# =====================================================================
# 5. DATA SPLITTING (Train-Test Split)
# =====================================================================
print("--- Step 5: Slicing into Training and Testing Sets ---")
# Separate your features (X) from the target classification column (y)
X = reduced_df.drop(columns=["income"])
y = reduced_df["income"]

# Perform an 80/20 train-test split on your single processed dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}")
# =====================================================================
# --- FINAL CLEANED, TRANSFORMED, AND REDUCED DATASET PREVIEW ---
# =====================================================================
print("\n" + "="*70)
print("--- FINAL PROCESSED SPLITS FOR TRAINING AND TESTING ---")
print("="*70)

# 1. Preview the Final Training Features (X_train) and Target (y_train) Side-by-Side
print(f"\n[TRAINING SET PREVIEW] - Total Samples: {X_train.shape[0]}")
print("-" * 50)
# Temporarily join them just for a clean layout printout
final_train_preview = X_train.copy()
final_train_preview['income'] = y_train

# Showing binary factors, engineered log metrics, and the final PCA components
preview_cols = ['income', 'gender', 'capital_net_log', 'Demographic_PCA_1', 'Demographic_PCA_2']
print(final_train_preview[preview_cols].head())


print("\n" + "-" * 50)


# 2. Preview the Final Testing Features (X_test) and Target (y_test) Side-by-Side
print(f"[TESTING SET PREVIEW] - Total Samples: {X_test.shape[0]}")
print("-" * 50)
final_test_preview = X_test.copy()
final_test_preview['income'] = y_test

print(final_test_preview[preview_cols].head())
print("="*70)

--- Step 1: Loading Dataset ---
Original Dataset Shape: (48842, 15)

--- Step 2: Cleaning the Data ---
Actual missing values detected per column:
workclass         2799
occupation        2809
native-country     857
dtype: int64

--- Step 3: Transforming Data ---
Data transformation complete.

--- Step 4: Reducing Data ---
Shape after manual feature selection: (48842, 41)
Final reduced dataframe shape: (48842, 40)

--- Step 5: Slicing into Training and Testing Sets ---
Training Features Shape: (39073, 39)
Testing Features Shape: (9769, 39)

--- FINAL PROCESSED SPLITS FOR TRAINING AND TESTING ---

[TRAINING SET PREVIEW] - Total Samples: 39073
--------------------------------------------------
       income  gender  capital_net_log  Demographic_PCA_1  Demographic_PCA_2
34342       0       0              0.0          -0.622872           2.523090
18559       0       1              0.0          -3.266216          -0.442860
12477       0       0              0.0          -0.615711          -0